# Demo 2 — A2A

**What changes from Demo 1:** the agentic loop moves *inside* an agent we can call over HTTP. The caller sends one sentence; the agent plans, fetches, writes a brief.

```
   you (this notebook)
        │  one sentence ("Analyze NVDA ...")
        │  HTTP + JSON-RPC
        ▼
   ┌─────────────────────────────────────┐
   │  analyst_agent.py  (A2A server)     │
   │  port 9999                          │
   │                                     │
   │   internal Claude loop ──┐          │
   │                          │          │
   │                          ▼          │
   │   stock_mcp_server.py (subprocess)  │
   └─────────────────────────────────────┘
        │
        ▼  written brief
   you (notebook prints it)
```

Agent: [`analyst_agent.py`](analyst_agent.py) — A2A server with an internal Claude + stock-MCP loop
Helpers: [`a2a_helpers.py`](a2a_helpers.py)

## First-time setup

```bash
python3.12 -m venv .venv && source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

Use the same `.env` from Demo 1 (the agent reads `ANTHROPIC_*` to call Claude internally).

In [ ]:
# ── Bootstrap: resolve paths to shared/ and sibling helpers ──
import sys
from pathlib import Path
HERE = Path.cwd()
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))   # so `from shared.X import Y` works
sys.path.insert(0, str(HERE))   # so sibling helpers import directly
STOCK_MCP_SERVER = str(ROOT / "shared" / "stock_mcp_server.py")


In [ ]:
%pip install -q "a2a-sdk<1.0" uvicorn httpx anthropic mcp yfinance python-dotenv

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
print("ready")

## Agent card

Self-description served at `/.well-known/agent-card.json`. Any A2A client can discover this agent through it.

In [ ]:
from analyst_agent import build_agent_card
print(build_agent_card().model_dump_json(indent=2, exclude_none=True))

## Start the agent (port 9999)

In [ ]:
from a2a_helpers import (
    start_agent, stop_agent, fetch_agent_card,
    print_agent_card, send_message, show_response,
)

agent = start_agent(str(HERE / "analyst_agent.py"))

## Discover it over HTTP

In [ ]:
print_agent_card(fetch_agent_card())

## Delegate a high-level task

We send **one sentence** — no tool list, no plan. The agent's internal Claude loop figures out which data to fetch and how to summarize.

In [ ]:
response = send_message("Analyze NVDA — recent price action, what the company does, and any notable news.")
show_response(response)

## Raw JSON-RPC envelope (optional)

Note `result.kind` and `result.status` — A2A models every call as a task with a lifecycle.

In [ ]:
show_response(response, raw=True)

In [ ]:
stop_agent(agent)